In [ ]:

from transformers import AutoModelForCausalLM
from peft import PeftConfig, PeftModel

# Base 모델 로드
base_model   = AutoModelForCausalLM.from_pretrained("beomi/KoAlpaca-Polyglot-5.8B")

# Adapter 설정 로드
adapter_path = "SiniDSBA/chatbot-koalpaca-polyglot-5_8b-5150step-1batch_0.7epoch"
peft_config  = PeftConfig.from_pretrained(adapter_path)

# PeftModel 생성
peft_model = PeftModel.from_pretrained(
    base_model,         # Base 모델 객체
    adapter_path,       # 어댑터 경로
    config=peft_config  # 키워드 인자로 PeftConfig 전달
)


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftConfig, PeftModel

# A. Base 모델 + 토크나이저
BASE_MODEL   = "beomi/KoAlpaca-Polyglot-5.8B"
tokenizer    = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model   = AutoModelForCausalLM.from_pretrained(BASE_MODEL)



Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
# B. Adapter-only repo
ADAPTER_REPO = "SiniDSBA/chatbot-koalpaca-polyglot-5_8b-5150step-1batch_0.7epoch"
peft_config  = PeftConfig.from_pretrained(ADAPTER_REPO)

# C. PeftModel 래핑
peft_model  = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
    config=peft_config,
)

# D. 어댑터 병합
merged_model = peft_model.merge_and_unload()


In [9]:
merged_model.save_pretrained("merged_model")
tokenizer.save_pretrained('merged_model')

('merged_model/tokenizer_config.json',
 'merged_model/special_tokens_map.json',
 'merged_model/tokenizer.json')

In [10]:
merged_model.push_to_hub("SiniDSBA/chatbot_test")
tokenizer.push_to_hub('SiniDSBA/chatbot_test')

model-00002-of-00005.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.25G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/SiniDSBA/chatbot_test/commit/8521a8568eae01231852d685ac81bdcef75708df', commit_message='Upload tokenizer', commit_description='', oid='8521a8568eae01231852d685ac81bdcef75708df', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SiniDSBA/chatbot_test', endpoint='https://huggingface.co', repo_type='model', repo_id='SiniDSBA/chatbot_test'), pr_revision=None, pr_num=None)

In [ ]:
# E. 사용 예시
input_ids = tokenizer("안녕하세요?", return_tensors="pt").input_ids.to(merged_model.device)
output    = merged_model.generate(input_ids, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))
# 3. (선택) 어댑터 병합 및 언로드 — 완전 통합 모델로 저장하고 싶을 때


In [ ]:
merged_model.save_pretrained("merged_model")
tokenizer.save_pretrained('merged_model')